In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 53.5 MB/s eta 0:00:00


In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

In [5]:
PROJECT_PATH = Path(
    "/content/drive/MyDrive/uterine-emg-rag"
)

EMBEDDINGS_PATH = PROJECT_PATH / "embeddings"
FAISS_PATH = PROJECT_PATH / "faiss_index"

In [6]:
metadata = pd.read_csv(
    EMBEDDINGS_PATH / "chunk_metadata.csv"
)

In [7]:
index = faiss.read_index(
    str(FAISS_PATH / "uterine_emg.index")
)

In [8]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [32]:
evaluation_queries = [
    {
        "query": "What are the characteristics of uterine EMG signals?",
        "relevant_papers": [
            "paper1.pdf",
            "paper2.pdf",
            "paper3.pdf",
            "paper4.pdf",
            "paper5.pdf"
        ]
    },
    {
        "query": "How can uterine EMG predict preterm labor?",
        "relevant_papers": [
            "paper1.pdf",
            "paper2.pdf",
            "paper3.pdf",
            "paper4.pdf",
            "paper5.pdf"
        ]
    },
    {
        "query": "What signal processing methods are used for uterine EMG?",
        "relevant_papers": [
            "paper1.pdf",
            "paper2.pdf",
            "paper3.pdf",
            "paper4.pdf",
            "paper5.pdf"
        ]
    }
]

In [33]:
from pathlib import Path

def normalize_paper_name(name):
    return Path(str(name)).stem.lower().strip()

In [34]:
print(normalize_paper_name("paper2.pdf"))

paper2


In [35]:
def retrieve(query, model, index, metadata, k=5):

    query_embedding = model.encode(
        [query]
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for rank, idx in enumerate(indices[0], start=1):

        results.append({
            "rank": rank,
            "index": int(idx),
            "score": float(scores[0][rank - 1]),
            "paper": metadata.iloc[idx]["paper"],
            "chunk_id": metadata.iloc[idx]["chunk_id"],
            "text": metadata.iloc[idx]["text"]
        })

    return results

In [36]:
query = evaluation_queries[0]["query"]

results = retrieve(
    query,
    model,
    index,
    metadata,
    k=5
)

In [37]:
for result in results:

    print(
        f"\nRank: {result['rank']}"
        f"\nScore: {result['score']:.4f}"
        f"\nPaper: {result['paper']}"
        f"\nChunk: {result['chunk_id']}"
    )

    print(result["text"][:500])


Rank: 1
Score: 0.7209
Paper: paper2
Chunk: 44
invasive EMG is crucial for interpreting surface recordings and for developing models of
uterine excitation and propagation.
3.2. Physiological Basis of Uterine EMG vs. Skeletal Muscle EMG
Electromyography in the classical sense often refers to recording electrical activity
from skeletal muscle. Uterine muscle, being smooth muscle, differs significantly in its
electrophysiological behavior. These distinctions shape both the waveform characteristics
and the interpretation of uterine EMG.
Excitati

Rank: 2
Score: 0.7209
Paper: paper2
Chunk: 41
EMG, the physiological distinctions between uterine and skeletal muscle activity, and the
technical methods used to acquire and interpret uterine EMG signals. In current clinical
practice, “uterine EMG” is effectively synonymous with EHG; the invasive and multiscale
material below is included to ground interpretation of the surface signal, not as a separate
clinical modality.
3.1. Historical Developmen

In [38]:
def precision_at_k(results, relevant_papers, k=5):

    relevant_papers = {
        normalize_paper_name(p)
        for p in relevant_papers
    }

    retrieved = results[:k]

    relevant_count = sum(
        normalize_paper_name(r["paper"]) in relevant_papers
        for r in retrieved
    )

    return relevant_count / k

In [39]:
score = precision_at_k(
    results,
    evaluation_queries[0]["relevant_papers"],
    k=5
)

print(score)

1.0


In [40]:
def recall_at_k(results, relevant_papers, k=5):

    relevant_papers = {
        normalize_paper_name(p)
        for p in relevant_papers
    }

    retrieved_papers = {
        normalize_paper_name(r["paper"])
        for r in results[:k]
    }

    relevant_retrieved = (
        retrieved_papers.intersection(relevant_papers)
    )

    return (
        len(relevant_retrieved)
        / len(relevant_papers)
    )

In [41]:
def reciprocal_rank(results, relevant_papers):

    relevant_papers = {
        normalize_paper_name(p)
        for p in relevant_papers
    }

    for result in results:

        if normalize_paper_name(result["paper"]) in relevant_papers:
            return 1 / result["rank"]

    return 0.0

In [42]:
evaluation_results = []

for item in evaluation_queries:

    query = item["query"]
    relevant_papers = item["relevant_papers"]

    results = retrieve(
        query,
        model,
        index,
        metadata,
        k=5
    )

    precision = precision_at_k(
        results,
        relevant_papers,
        k=5
    )

    recall = recall_at_k(
        results,
        relevant_papers,
        k=5
    )

    rr = reciprocal_rank(
        results,
        relevant_papers
    )

    evaluation_results.append({
        "query": query,
        "precision@5": precision,
        "recall@5": recall,
        "MRR": rr
    })

In [43]:
evaluation_df = pd.DataFrame(
    evaluation_results
)

evaluation_df

,query,precision@5,recall@5,MRR
0,What are the characteristics of uterine EMG si...,1.0,0.2,1.000000
1,How can uterine EMG predict preterm labor?,0.4,0.4,0.333333
2,What signal processing methods are used for ut...,1.0,0.2,1.000000


In [45]:
print(
    "Mean Precision@5:",
    evaluation_df["precision@5"].mean()
)

print(
    "Mean Recall@5:",
    evaluation_df["recall@5"].mean()
)

print(
    "MRR:",
    evaluation_df["MRR"].mean()
)

Mean Precision@5: 0.7999999999999999
Mean Recall@5: 0.26666666666666666
MRR: 0.7777777777777777


In [46]:
EVALUATION_PATH = PROJECT_PATH / "evaluation"

EVALUATION_PATH.mkdir(exist_ok=True)

In [47]:
evaluation_df.to_csv(
    EVALUATION_PATH / "retrieval_evaluation.csv",
    index=False
)